In [12]:
import k2
import torch
import numpy as np
import pynini
from pynini.lib import pynutil
import io
import os
from glob import glob
import random
from collections import defaultdict

In [13]:

# --- 1. Setup CUDA Device ---
# Check if CUDA is available and set the device.
if torch.cuda.is_available():
    DEVICE = torch.device("cuda", 0)
    print(f"CUDA device found: {torch.cuda.get_device_name(0)}")
else:
    DEVICE = torch.device("cpu")
    print("CUDA not available. Running on CPU (install CUDA/NVIDIA drivers for GPU acceleration).")

CUDA device found: NVIDIA GeForce RTX 4090


In [14]:
def fst_to_arc_dict(fst_pynini):
    arcs = defaultdict(list)
    aux_labels = defaultdict(list)
    scores = defaultdict(list)
    states = list(fst_pynini.states())
    final_state = max(states) + 1
    for state_id in states:
        for arc in fst_pynini.arcs(state_id):
            arcs[state_id].append([state_id, arc.nextstate, arc.ilabel, 0])
            aux_labels[state_id].append(arc.olabel)
            scores[state_id].append(float(arc.weight))
            if fst_pynini.final(arc.nextstate) != pynini.Weight.zero('tropical'):
                arcs[arc.nextstate].append(
                    [arc.nextstate, final_state, -1, 0]
                )
                aux_labels[arc.nextstate].append(-1)
                scores[arc.nextstate].append(float(fst_pynini.final(arc.nextstate)))
    
    arclist = []
    auxlabellist = []
    scorelist = []
    for state_id in states:
        arclist.extend(arcs[state_id])
        auxlabellist.extend(aux_labels[state_id])
        scorelist.extend(scores[state_id])


    arcs_tensor = torch.tensor(arclist, dtype=torch.int32)
    aux_labels_tensor = torch.tensor(auxlabellist, dtype=torch.int32)
    arc_dict = {
        'arcs': arcs_tensor,
        'aux_labels': aux_labels_tensor,
        'scores': torch.tensor(scorelist, dtype=torch.float32),
    }
    return arc_dict

In [15]:
def k2_fst(fst_pynini):
    arc_dict = fst_to_arc_dict(fst_pynini)
    scores = arc_dict.pop('scores')
    k2_fst_obj = k2.Fsa.from_dict(arc_dict).to(DEVICE)
    k2_fst_obj.scores = scores.to(DEVICE)
    return k2_fst_obj

boy = pynini.cross("cat", "dog") | pynini.cross("mouse", "cheese")
k2_fst(boy)

In [16]:
from src.fst_helpers import fst, get_lattice_strs, extract_word_from_labels
from src.parser import get_main_parser

boy = fst("àprí", weight=0.5)
lmzr, alzr, inflr = get_main_parser()

get_lattice_strs(boy@lmzr)

['àprí[case=unmarked][number=unmarked][aux=unmarked][final_lowering=unmarked][fv=unmarked][left_h=unmarked][part_of_speech=noun]',
 'àprí[case=nominative][number=singular][aux=unmarked][final_lowering=unmarked][fv=unmarked][left_h=unmarked][part_of_speech=noun]']

In [17]:
boy_k2 = k2_fst(boy)
lmzr_k2 = k2_fst(lmzr)

# k2.compose needs both inputs to be on CPU
boy_k2 = k2.arc_sort(boy_k2)
lmzr_k2 = k2.arc_sort(lmzr_k2)
lattice = k2.compose(boy_k2.to('cpu'), lmzr_k2.to('cpu'))

In [18]:
# unless we pass treat_epsilons_specially=False
boy_k2 = k2.add_epsilon_self_loops(boy_k2)
lmzr_k2 = k2.add_epsilon_self_loops(lmzr_k2)
boy_k2 = k2.arc_sort(boy_k2)
lmzr_k2 = k2.arc_sort(lmzr_k2)
lattice = k2.compose(boy_k2, lmzr_k2, treat_epsilons_specially=False)
lattice = k2.remove_epsilon_self_loops(lattice)

In [ ]:
def decode_label_tensor(label_tensor):
    labels = label_tensor.tolist()
    words = []
    for label in labels:
        if label == -1:
            words.append("<eps>")
        else:
            words.append(pynini.SymbolTable.read_text("data/symbols.txt").find(label))
    return words

In [20]:
lattice_batch = k2.create_fsa_vec([lattice, lattice])
best_path = k2.shortest_path(lattice_batch, use_double_scores=True)
best_path.aux_labels, best_path.num_arcs, extract_word_from_labels(best_path.aux_labels)

(tensor([     0,     61,     63,     24,     48,     51,     62, 983080, 983083,
         983104, 983107, 983096, 983109, 983087,     -1,      0,     61,     63,
             24,     48,     51,     62, 983080, 983083, 983104, 983107, 983096,
         983109, 983087,     -1], device='cuda:0', dtype=torch.int32),
 30,
 'àprí[case=unmarked][number=unmarked][aux=unmarked][final_lowering=unmarked][fv=unmarked][left_h=unmarked][part_of_speech=noun]àprí[case=unmarked][number=unmarked][aux=unmarked][final_lowering=unmarked][fv=unmarked][left_h=unmarked][part_of_speech=noun]')